<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Binance

Manages the Binance data download and processing using the [ccxt](https://github.com/ccxt/ccxt) package.
Candles (OHLCV) are downloaded through `ccxt.binance` and saved, one file per trading pair, into a
folder named after the exchange (e.g. `../data/binance`).

Notes:

- The main Binance API is **geo-blocked in some regions (e.g. the US)**. Use `binance_available()`
  to check reachability; the live test cells in this notebook skip gracefully when the API cannot
  be reached. From the US you can pass `exchange_id='binanceus'` to the functions below.
- Binance serves deep history (up to 1000 candles per request), so full backfills are possible.
- Rate limits are handled by ccxt (`enableRateLimit=True`).

** Finally, datetime columns are in UTC. **

In [0]:
#| echo: false
#| output: asis
show_doc(binance_available)

---

### binance_available

```python
def binance_available(
    exchange_id:str='binance', timeout:int=10000
):
```

*Check whether the Binance API is reachable from this machine.*

The main Binance API is geo-blocked in some regions (e.g. the US, HTTP 451).

Args:
    exchange_id (str, optional): ccxt exchange id - "binance" or "binanceus".
        Defaults to "binance"
    timeout (int, optional): Request timeout in milliseconds. Defaults to 10000

Returns:
    bool: True if the API responds, False otherwise (geo-block, network error, etc.)

In [0]:
#| echo: false
#| output: asis
show_doc(retry_fetch_ohlcv)

---

### retry_fetch_ohlcv

```python
def retry_fetch_ohlcv(
    exchange, max_retries, symbol, timeframe, since, limit, verbose:bool=False
):
```

*Fetch a single page of OHLCV candles from an exchange, retrying on failure.*

Args:
    exchange (ccxt.Exchange): Instantiated ccxt exchange object
    max_retries (int): Maximum number of retries before raising the last error
    symbol (str): Trading pair symbol (e.g. "BTC/USDT")
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
    since (int): Start time in milliseconds since epoch (UTC)
    limit (int): Maximum number of candles to fetch
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]

Raises:
    Exception: The last ccxt error if all retries fail

In [0]:
#| echo: false
#| output: asis
show_doc(scrape_ohlcv)

---

### scrape_ohlcv

```python
def scrape_ohlcv(
    exchange, symbol, timeframe, since, end:NoneType=None, max_retries:int=3, limit:int=1000, verbose:bool=False
):
```

*Download OHLCV candles in pages of `limit` bars between `since` and `end`.*

Args:
    exchange (ccxt.Exchange): Instantiated ccxt exchange object (markets loaded)
    symbol (str): Trading pair symbol (e.g. "BTC/USDT")
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
    since (int or str): Start time in milliseconds since epoch or ISO 8601 string
    end (int or str, optional): End time in milliseconds or ISO 8601 string.
        Defaults to now.
    max_retries (int, optional): Retries per page. Defaults to 3
    limit (int, optional): Candles per request. Defaults to 1000 (Binance maximum)
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]

In [0]:
#| echo: false
#| output: asis
show_doc(ohlcv_to_df)

---

### ohlcv_to_df

```python
def ohlcv_to_df(
    ohlcv, symbol
):
```

*Convert a raw ccxt OHLCV list into a tidy DataFrame.*

Args:
    ohlcv (list): List of candles [timestamp_ms, open, high, low, close, volume]
    symbol (str): Trading pair symbol added as the `pair` column

Returns:
    pandas.DataFrame: DataFrame with columns:
        - datetime: Candle timestamp (UTC, timezone-aware)
        - open, high, low, close, volume: OHLCV values
        - pair: Trading pair symbol
    Sorted by datetime with duplicate timestamps removed.

In [0]:
#| echo: false
#| output: asis
show_doc(binance_ohlcv)

---

### binance_ohlcv

```python
def binance_ohlcv(
    symbol:str='BTC/USDT', timeframe:str='1h', since:NoneType=None, end:NoneType=None, max_retries:int=3,
    limit:int=1000, exchange:NoneType=None, exchange_id:str='binance', verbose:bool=False
):
```

*Download OHLCV candles for a symbol from Binance.*

Args:
    symbol (str, optional): Trading pair symbol. Defaults to "BTC/USDT"
    timeframe (str, optional): Candle timeframe. Defaults to "1h"
    since (int or str, optional): Start time (ms since epoch or ISO 8601).
        If None, downloads the most recent ~`limit` candles.
    end (int or str, optional): End time (ms or ISO 8601). Defaults to now
    max_retries (int, optional): Retries per page. Defaults to 3
    limit (int, optional): Candles per request. Defaults to 1000
    exchange (ccxt.Exchange, optional): Reusable exchange instance. If None, a new one is created
    exchange_id (str, optional): ccxt exchange id used when `exchange` is None -
        "binance" or "binanceus". Defaults to "binance"
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    pandas.DataFrame: Tidy OHLCV DataFrame (see `ohlcv_to_df`)

In [0]:
#| echo: false
#| output: asis
show_doc(binance_usdt_tokens)

---

### binance_usdt_tokens

```python
def binance_usdt_tokens(
    exchange:NoneType=None, exchange_id:str='binance'
):
```

*Retrieves all active Binance spot trading pairs quoted in USDT.*

Args:
    exchange (ccxt.Exchange, optional): Reusable exchange instance. If None, a new one is created
    exchange_id (str, optional): ccxt exchange id used when `exchange` is None -
        "binance" or "binanceus". Defaults to "binance"

Returns:
    pandas.DataFrame: DataFrame with columns:
        - id: Binance market id (e.g. 'BTCUSDT')
        - symbol: Unified ccxt symbol (e.g. 'BTC/USDT')
        - base: Base currency (e.g. 'BTC')
        - quote: Always 'USDT' for this filtered dataset
        - active: Whether the market is currently active

#### Example / tests

The live tests below run only if the Binance API is reachable from this machine
(it is geo-blocked in some regions, e.g. the US). Offline tests always run.

In [ ]:
#|eval: false
# Offline test: ohlcv_to_df shapes raw candles correctly and removes duplicates
sample = [[1700000000000, 1.0, 2.0, 0.5, 1.5, 10.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0]]  # duplicate on purpose
df_sample = ohlcv_to_df(sample, 'TEST/USDT')
assert len(df_sample) == 2
assert list(df_sample.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert str(df_sample['datetime'].dt.tz) == 'UTC'
df_sample

In [ ]:
#|eval: false
# Live test (skips gracefully if Binance is geo-blocked / unreachable)
BINANCE_OK = binance_available()
if BINANCE_OK:
    start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ')
    df_test = binance_ohlcv(symbol='BTC/USDT', timeframe='1h', since=start)
    assert not df_test.empty
    assert list(df_test.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
    assert df_test['datetime'].dt.tz is not None
    assert df_test['datetime'].is_monotonic_increasing
    assert (df_test['pair'] == 'BTC/USDT').all()
    assert len(df_test) > 24
    print(df_test.tail())
else:
    print('Skipping live Binance OHLCV test (API unreachable, likely geo-blocked)')

In [ ]:
#|eval: false
# Live test: token universe contains the major pairs (skips if unreachable)
if BINANCE_OK:
    tokens = binance_usdt_tokens()
    assert not tokens.empty
    assert 'BTC/USDT' in tokens['symbol'].tolist()
    assert (tokens['quote'] == 'USDT').all()
    print(tokens.head())
else:
    print('Skipping live Binance token universe test (API unreachable, likely geo-blocked)')

In [0]:
#| echo: false
#| output: asis
show_doc(save_file)

---

[source](https://github.com/silvaac/token_data/blob/main/token_data/coinbase.py#L175){target="_blank" style="float:right; font-size:smaller"}

### save_file

```python
def save_file(
    df, folder_path, file_name, type:str='parquet'
):
```

*Save a pandas DataFrame to a file in either CSV or Parquet format.*

Args:
    df (pandas.DataFrame): The DataFrame to save
    folder_path (str): Directory path where the file will be saved
    file_name (str): Name of the file without extension
    type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

The function saves the DataFrame to the specified path, handling the file extension automatically.
For CSV files, the index is not saved. Creates the folder if it doesn't exist.

In [0]:
#| echo: false
#| output: asis
show_doc(file_name_to_symbol)

---

### file_name_to_symbol

```python
def file_name_to_symbol(
    file_name
):
```

*Convert a file name back into a ccxt symbol.*

Example: 'BTC-USDT_1h.parquet' -> 'BTC/USDT'

In [0]:
#| echo: false
#| output: asis
show_doc(symbol_to_file_name)

---

### symbol_to_file_name

```python
def symbol_to_file_name(
    symbol, timeframe:str='1h'
):
```

*Convert a ccxt symbol and timeframe into a file name (without extension).*

Example: ('BTC/USDT', '1h') -> 'BTC-USDT_1h'

In [ ]:
#|eval: false
# Offline test: save_file round-trip and file-name helpers
import tempfile
tmp_dir = tempfile.mkdtemp()
save_file(df_sample, tmp_dir, 'TEST-USDT_1h', type='parquet')
df_back = pd.read_parquet(f"{tmp_dir}/TEST-USDT_1h.parquet")
assert df_back.shape == df_sample.shape
assert list(df_back.columns) == list(df_sample.columns)
assert symbol_to_file_name('BTC/USDT', '1h') == 'BTC-USDT_1h'
assert file_name_to_symbol('BTC-USDT_1h.parquet') == 'BTC/USDT'
print('save_file round-trip OK')

In [0]:
#| echo: false
#| output: asis
show_doc(binance_to_file)

---

### binance_to_file

```python
def binance_to_file(
    folder_path:str='../data/binance', token_list:list=['BTC/USDT', 'ETH/USDT'], type:str='parquet',
    timeframe:str='1h', refresh_hours:int=24, first_date:str='2021-01-01T00:00:00Z', all_tokens:bool=True,
    pause:int=1, exchange_id:str='binance', verbose:bool=False
):
```

*Downloads and maintains historical Binance OHLCV data, saving one file per pair.*

Args:
    folder_path (str): Path where pair data files will be stored, named after the
        exchange. Defaults to "../data/binance"
    token_list (list): List of ccxt symbols to process. Defaults to ['BTC/USDT', 'ETH/USDT']
    type (str): File format to save data - either "csv" or "parquet". Defaults to "parquet"
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d"). Defaults to "1h"
    refresh_hours (int): Hours of the most recent data to re-download when updating.
        Defaults to 24
    first_date (str): ISO 8601 start date used for the initial full-history download
        of a pair. Defaults to '2021-01-01T00:00:00Z'
    all_tokens (bool): If True, includes any additional pairs found in the folder path.
        Defaults to True
    pause (int): Seconds to wait between pairs. Defaults to 1
    exchange_id (str): ccxt exchange id - "binance" or "binanceus". Defaults to "binance"
    verbose (bool): If True, prints download progress. Defaults to False

The function:
- Creates the folder_path if it doesn't exist
- Date/Time is UTC
- For each pair, checks if a data file exists:
    - If exists: Loads the file and appends new data, refreshing the last `refresh_hours`
    - If not exists: Downloads full history starting from `first_date`
- Saves data in the specified format, handling duplicates and sorting by datetime

#### Example / tests

In [ ]:
#|eval: false
# Live test: download BTC/USDT into a temporary exchange folder, then re-run incrementally
# (skips gracefully if Binance is geo-blocked / unreachable)
import tempfile
if BINANCE_OK:
    binance_dir = tempfile.mkdtemp()
    recent = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=3)).strftime('%Y-%m-%dT%H:%M:%SZ')
    binance_to_file(folder_path=binance_dir, token_list=['BTC/USDT'], type='parquet',
                    timeframe='1h', first_date=recent, pause=0)
    assert os.path.exists(f"{binance_dir}/BTC-USDT_1h.parquet")
    df1 = pd.read_parquet(f"{binance_dir}/BTC-USDT_1h.parquet")
    n1 = len(df1)
    assert n1 > 0
    # Incremental re-run must not shrink the file and must not create duplicates
    binance_to_file(folder_path=binance_dir, token_list=['BTC/USDT'], type='parquet',
                    timeframe='1h', first_date=recent, pause=0)
    df2 = pd.read_parquet(f"{binance_dir}/BTC-USDT_1h.parquet")
    assert len(df2) >= n1
    assert not df2['datetime'].duplicated().any()
    print(f"First run: {n1} rows, second run: {len(df2)} rows")
else:
    print('Skipping live binance_to_file test (API unreachable, likely geo-blocked)')

In [ ]:
#|eval: false
# Download / update a set of USDT pairs into the exchange-named data folder
# For all USDT pairs use: token_list = binance_usdt_tokens()['symbol'].tolist()
# From the US, pass exchange_id='binanceus' (and adjust folder_path if desired)
binance_to_file(folder_path="../data/binance",
                token_list=['BTC/USDT', 'ETH/USDT', 'SOL/USDT'],
                type="parquet", timeframe='1h')